# Wi-Fi/Cellular UAV Mobility Prediction and Trajectory Optimization
Full Colab execution notebook. Runtime -> Change runtime type -> GPU before running.

## Phase 0: Get the code
Clones the repo and installs it as a package so every `!python scripts/...` cell can import `wifi_uav`. Set `PRESET` to `smoke` for a 2-minute check, `medium` for a quick result, or `full` for the final results (GPU recommended).

In [ ]:
import os
REPO_URL = 'https://github.com/sahilsingh069/wifi-uav-project.git'
PROJECT_ROOT = '/content/wifi-uav-project'
PRESET = 'full'  # 'smoke' | 'medium' | 'full'
if not os.path.isdir(PROJECT_ROOT):
    !git clone -q {REPO_URL} {PROJECT_ROOT}
os.chdir(PROJECT_ROOT)
!pip install -q -e .
print(os.getcwd(), PRESET)

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## Phase 1: RF dataset generation (3GPP UMi/UMa channel, correlated shadowing)

In [ ]:
!python scripts/phase1_collect_data.py --preset {PRESET}

## Phase 2: Feature extraction (split by episode to avoid leakage)

In [ ]:
!python scripts/phase2_build_features.py --preset {PRESET}

## Phase 3: LSTM-Transformer training

In [ ]:
!python scripts/phase3_train_predictor.py --preset {PRESET}

## Phase 4: UAV environment sanity check (UAVs observe predicted user positions)

In [ ]:
!python scripts/phase4_sanity_env.py --preset {PRESET}

## Phase 5: MADDPG training

In [ ]:
!python scripts/phase5_train_maddpg.py --preset {PRESET}

## Phase 6: Evaluation against baselines

In [ ]:
!python scripts/phase6_evaluate.py --preset {PRESET}

In [ ]:
from IPython.display import Image, display
for name in ['training_curves', 'baseline_comparison', 'coverage_heatmap']:
    path = f'results/{name}.png'
    if os.path.exists(path):
        display(Image(path))

In [ ]:
!zip -qr wifi_uav_results.zip data/normalization.npz checkpoints results